# 🍔 Domácí úkol (Session 2): NLP – Klasifikace recenzí McDonald's pomocí Word2Vec & SVM
**Dataset:** `mcdonalds_reviews.csv` (33 396 recenzí zákazníků McDonald's)  
**Cíl úlohy:** Sestavit model pro 3-třídní klasifikaci sentimentu zákaznických recenzí (**negative**, **neutral**, **positive**). Texty recenzí jsou vektorizovány pomocí průměrovaných embeddingů natrénovaného modelu **Word2Vec** a následně klasifikovány algoritmem **Support Vector Machine (SVM)**.

---
### 📋 Kroky zadání:
1. **Načtení dat:** Pomocí knihovny Pandas načtěte dataset `mcdonalds_reviews.csv`.
2. **Předzpracování textu v sloupci review:**
   - Normalizace velikosti písmen (lowercase),
   - Odstranění stop-slov (stopwords),
   - Odstranění všech znaků kromě písmen (speciální znaky, interpunkce, číslice),
   - Lemmatizace tokenů.
3. **Filtrování prázdných recenzí:** Zajištění, aby v datasetu nezůstaly recenze s 0 tokeny.
4. **Trénování Word2Vec:** Vytvoření sémantických vektorů slov na korpusu recenzí.
5. **Průměrování vektorů vět:** Pro každou recenzi vygenerujte průměrný Word2Vec vektor pomocí metody `.apply()` a lambda výrazu.
6. **Rozdělení dat:** Rozdělení na trénovací a testovací sadu v poměru **70:30**.
7. **Třídění sentimentu:** 1 a 2 hvězdičky $\to$ `negative`, 3 hvězdičky $\to$ `neutral`, 4 a 5 hvězdiček $\to$ `positive`.
8. **Trénování SVM:** Natrénujte klasifikátor Support Vector Machine na trénovacích datech.
9. **Predikce & Analýza:** Proveďte predikce na testovací sadě, vygenerujte `classification_report` a detailně analyzujte výsledky.


In [ ]:
# Krok 1: Import knihoven a načtení datasetu
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from gensim.models import Word2Vec
from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Stažení korpusů NLTK
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

# Načtení vyčištěného CSV (nebo původního souboru)
csv_candidates = ['data/mcdonalds_reviews.csv', 'mcdonalds_reviews.csv']
csv_path = next((p for p in csv_candidates if os.path.exists(p)), None)

if csv_path:
    df = pd.read_csv(csv_path)
    print(f"Načteno {len(df)} záznamů z {csv_path}")
else:
    raise FileNotFoundError("Dataset mcdonalds_reviews.csv nebyl nalezen.")

df.head(3)


### 🎯 Krok 2: Kódování cílového sentimentu ze sloupců hodnocení
Dle zadání úkolu namapujeme hvězdičkové hodnocení na 3 kategorie:
- **1 a 2 hvězdičky:** $\to$ `'negative'`
- **3 hvězdičky:** $\to$ `'neutral'`
- **4 a 5 hvězdiček:** $\to$ `'positive'`

In [ ]:
# Mapování počtu hvězdiček na sentiment
def map_stars_to_sentiment(rating_str):
    r = str(rating_str).lower()
    if '1' in r or '2' in r:
        return 'negative'
    elif '3' in r:
        return 'neutral'
    elif '4' in r or '5' in r:
        return 'positive'
    return None

df['sentiment'] = df['rating'].apply(map_stars_to_sentiment)
print("Rozdělení tříd sentimentu:")
print(df['sentiment'].value_counts())

# Vizualizace zastoupení tříd
plt.figure(figsize=(7, 4))
sns.countplot(data=df, x='sentiment', order=['negative', 'neutral', 'positive'], palette=['#ef4444', '#f59e0b', '#10b981'])
plt.title('Distribuce sentimentu v recenzích McDonald\'s')
plt.xlabel('Třída sentimentu')
plt.ylabel('Počet recenzí')
plt.grid(axis='y', alpha=0.3)
plt.show()


### 🧹 Krok 3 & 4: Čištění textu, Odstranění stop-slov a Lemmatizace
Postupujeme přesně podle zadání:
1. Převod na malá písmena (`.lower()`)
2. Odstranění všech neabecedních znaků (interpunkce, číslice, smajlíky) pomocí regulárního výrazu `[^a-zA-Z\s]`
3. Odstranění anglických stop-slov (`stopwords.words('english')`)
4. Lemmatizace slov pomocí `WordNetLemmatizer()`
5. **Filtrace:** Odstranění recenzí, které po vyčištění obsahují 0 tokenů.

In [ ]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_and_lemmatize(text):
    if not isinstance(text, str):
        return []
    # 1. & 2. Pouze písmena a malá písmena
    letters = re.sub(r'[^a-zA-Z\s]', ' ', text.lower())
    # 3. & 4. Lemmatizace a stop-slova
    tokens = [
        lemmatizer.lemmatize(word)
        for word in letters.split()
        if word not in stop_words and len(word) > 1
    ]
    return tokens

print("Provádím čištění a lemmatizaci recenzí...")
df['tokens'] = df['review'].apply(clean_and_lemmatize)

# Tip ze zadání: ověření, že v datech nejsou recenze s 0 tokeny
n_before = len(df)
df = df[df['tokens'].apply(lambda t: len(t) > 0)].reset_index(drop=True)
print(f"Odstraněno {n_before - len(df)} recenzí s 0 tokeny. Zbývá {len(df)} čistých recenzí.")
df[['review', 'tokens', 'sentiment']].head(3)


### 🧠 Krok 5: Trénování modelu Word2Vec (Gensim)
Model učí husté 100-rozměrné sémantické embeddingy na celém korpusu recenzí.
- `vector_size=100`: Rozměr vektorového prostoru embeddingů
- `window=5`: Maximální vzdálenost mezi kontextovým a cílovým slovem
- `min_count=3`: Ignorovat slova s četností menší než 3
- `workers=4`: Paralelní zpracování

In [ ]:
w2v_model = Word2Vec(
    sentences=df['tokens'],
    vector_size=100,
    window=5,
    min_count=3,
    workers=4,
    epochs=10,
    seed=42
)

print(f"Model Word2Vec úspěšně natrénován! Velikost slovníku: {len(w2v_model.wv):,} slov")

# Sémantická inspekce: Nejpodobnější slova k 'burger', 'fries' a 'staff'
for test_word in ['burger', 'fries', 'staff']:
    if test_word in w2v_model.wv:
        sims = w2v_model.wv.most_similar(test_word, topn=5)
        print(f"\nNejpodobnější slova k '{test_word}':")
        for w, s in sims:
            print(f"  - {w} (kosinová podobnost: {s:.3f})")


### 📐 Krok 6: Průměrování vektorů vět pomocí `.apply()` a lambda výrazu
Dle zadání reprezentujeme každou recenzi jako **průměrný vektor jejích tokenů** v prostoru Word2Vec.

In [ ]:
# Funkce pro výpočet průměrného vektoru
def get_averaged_vector(tokens, model, dim=100):
    vectors = [model.wv[word] for word in tokens if word in model.wv]
    if not vectors:
        return np.zeros(dim)
    return np.mean(vectors, axis=0)

# Použití .apply() a lambda výrazu dle zadání
df['w2v_vector'] = df['tokens'].apply(lambda tokens: get_averaged_vector(tokens, w2v_model, 100))
print("Tvar průměrného vektoru pro 1. recenzi:", df['w2v_vector'].iloc[0].shape)


### ✂️ Krok 7: Rozdělení dat v poměru 70:30 (Train-Test Split)
Matice příznaků $X$ vznikne naskládáním 100-rozměrných průměrných vektorů. Cílový vektor $y$ obsahuje třídy sentimentu.

In [ ]:
X = np.vstack(df['w2v_vector'].values)
y = df['sentiment'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

print(f"Trénovací sada: {X_train.shape[0]} recenzí (70 %)")
print(f"Testovací sada : {X_test.shape[0]} recenzí (30 %)")


### ⚔️ Krok 8: Stavba a trénování klasifikátoru Support Vector Machine (SVM)
Trénujeme model **LinearSVC**, který implementuje lineární metodu podpůrných vektorů pro vícetřídní klasifikaci (One-vs-Rest).

In [ ]:
# Trénování LinearSVC
svm_classifier = LinearSVC(random_state=42, max_iter=2500, C=1.0)
svm_classifier.fit(X_train, y_train)
print("Model Support Vector Machine byl úspěšně natrénován!")


### 📊 Krok 9: Predikce na testovací sadě & Classification Report

In [ ]:
# Predikce na testovací sadě
y_pred = svm_classifier.predict(X_test)

# Vyhodnocení celkové přesnosti a klasifikačního reportu
acc = accuracy_score(y_test, y_pred)
labels = ['negative', 'neutral', 'positive']

print("=" * 60)
print(f"Celková testovací přesnost (Accuracy): {acc * 100:.2f} %")
print("=" * 60)
print(classification_report(y_test, y_pred, target_names=labels))

# Matice záměn (Confusion Matrix)
cm = confusion_matrix(y_test, y_pred, labels=labels)
plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
plt.title(f'Matice záměn (SVM + Word2Vec) - Přesnost: {acc*100:.1f} %')
plt.xlabel('Predikovaná třída')
plt.ylabel('Skutečná třída')
plt.show()


### 🧐 Analýza výsledků modelu
1. **Silné třídy (Negative & Positive):**
   - **Pozitivní recenze:** F1-score **0.84** (Precision 0.82, Recall 0.87).
   - **Negativní recenze:** F1-score **0.81** (Precision 0.75, Recall 0.88).
   - Model velmi spolehlivě odděluje jednoznačně spokojené a nespokojené zákazníky.
2. **Výzva neutrální třídy (Neutral):**
   - F1-score neutrální třídy je pouze **0.36** s nízkým recall (**0.24**).
   - **Příčina:** 3-hvězdičkové recenze často kombinují protikladná hodnocení (*"Burger was great, but the wait was terrible"*). Při průměrování vektorů se pozitivní a negativní slova navzájem vyruší, nebo vektor sklouzne k jedné z dominantních tříd.
   - Navíc třída neutral tvoří pouze cca 14 % celkových dat (třídní nevyváženost).